# 00 — Data Setup

**Este notebook é a fundação do projeto.** Execute-o antes de qualquer outro.

O que ele faz:
1. Baixa todos os arquivos do Anthropic Economic Index via `huggingface_hub`
2. Carrega as 3 releases com dados geográficos (Ago 2025, Nov 2025, Fev 2026)
3. Enriquece o dataframe com `country_name`, `release_date`, `usage_per_capita_index` (calculado para releases raw)
4. Exibe o dicionário de dados e metadados das variáveis
5. Salva o dataset consolidado em `data/aei_all_releases.parquet`

---
**Outputs gerados:**
- `data/aei_all_releases.parquet` — dataset completo, todas as releases
- `data/aei_aug2025.parquet` — Ago 2025 (enriched, com automation_pct etc.)
- `data/aei_feb2026.parquet` — Fev 2026 (mais recente, com variáveis novas)

In [ ]:
%pip install -q huggingface_hub pandas pyarrow

In [ ]:
import sys
import pandas as pd
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files
from IPython.display import display, HTML

# Paths
ROOT      = Path('..').resolve()
DATA_DIR  = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

# Importa metadados do projeto
sys.path.insert(0, str(ROOT / 'src'))
from metadata import RELEASES, BASE_COLUMNS, VARIABLES, COLLABORATION_PATTERNS, COUNTRY_LABELS, BRAZIL, COMPARABLES

REPO = 'Anthropic/EconomicIndex'

def dl(filename):
    return Path(hf_hub_download(repo_id=REPO, filename=filename,
                                repo_type='dataset', local_dir=DATA_DIR))

print('Setup ok. ROOT:', ROOT)

---
## 1. Download dos arquivos

In [ ]:
# Lista arquivos AEI disponíveis no repositório
all_files = list(list_repo_files(REPO, repo_type='dataset'))
aei_files = [f for f in all_files
             if ('aei_enriched_claude_ai' in f or 'aei_raw_claude_ai' in f)
             and f.endswith('.csv')]

print(f'{len(aei_files)} arquivo(s) AEI encontrado(s):')
for f in sorted(aei_files):
    print(f'  {f}')

In [ ]:
# Download: arquivo de população (necessário para normalização per capita)
POP_FILE = 'release_2025_09_15/data/input/working_age_pop_2024_country_raw.csv'
path_pop = dl(POP_FILE)
print('Pop file:', path_pop)

---
## 2. Mapa de nomes de países

In [ ]:
df_pop_raw = pd.read_csv(path_pop)

# Filtra apenas entidades que são países reais (ISO-3 de 3 letras maiúsculas sem agregados regionais)
# Agregados do Banco Mundial têm códigos como AFE, AFW, ARB, CSS, etc.
# Países reais: 3 letras, e existem no nosso dicionário de labels ou têm population > 0
country_name_map = (
    df_pop_raw[['countryiso3code', 'country.value']]
    .drop_duplicates()
    .set_index('countryiso3code')['country.value']
    .to_dict()
)

# Sobrescreve com nomes em português onde disponível
country_name_map.update({k: v for k, v in COUNTRY_LABELS.items()})

print(f'Países/regiões no mapa: {len(country_name_map)}')
print('Exemplos:')
for iso in ['BRA', 'USA', 'ARG', 'IND', 'ZAF']:
    print(f'  {iso} -> {country_name_map.get(iso, "N/A")}')

---
## 3. Carregamento de todas as releases

In [ ]:
# Normalização per capita: usage_per_capita_index = usage_pct / pop_share
pop_values = df_pop_raw.set_index('countryiso3code')['value'].to_dict()
total_pop  = sum(v for v in pop_values.values() if pd.notna(v) and v > 0)
pop_share  = {k: v / total_pop for k, v in pop_values.items() if pd.notna(v) and v > 0}

print(f'Total pop no dataset: {total_pop:,.0f}')
print(f'Países com pop_share: {len(pop_share)}')

frames = []

for release_key, meta in RELEASES.items():
    print(f"\nCarregando {meta['label']} ({meta['period']}) ...")
    raw = pd.read_csv(dl(meta['file']))

    # Colunas de contexto temporal
    raw['release_date']  = pd.Timestamp(meta['reference_dt'])
    raw['release_label'] = meta['label']
    raw['release_key']   = release_key
    raw['schema_type']   = meta['schema']

    # Coluna de nome do país
    raw['country_name'] = raw['geo_id'].map(country_name_map)

    # Computa usage_per_capita_index para releases raw (não vem no arquivo)
    if meta['schema'] == 'raw':
        mask = (
            (raw['geography'] == 'country') &
            (raw['facet']     == 'country') &
            (raw['variable']  == 'usage_pct')
        )
        idx_rows = raw[mask].copy()
        idx_rows['value']    = idx_rows.apply(
            lambda r: r['value'] / pop_share.get(r['geo_id'], float('nan')), axis=1
        )
        idx_rows['variable'] = 'usage_per_capita_index'
        raw = pd.concat([raw, idx_rows], ignore_index=True)

    frames.append(raw)
    vars_all = sorted(raw['variable'].unique())
    print(f'  {len(raw):,} linhas | {len(vars_all)} variáveis | países: {raw[raw["geography"]=="country"]["geo_id"].nunique()}')

df_all = pd.concat(frames, ignore_index=True)
print(f'\nDataframe consolidado: {df_all.shape}')
print(f'Releases: {df_all["release_label"].unique()}')

---
## 4. Validação e inspeção

In [ ]:
print('=== SHAPE E TIPOS ===')
print(df_all.dtypes)
print(f'\nMemória: {df_all.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
print('=== AMOSTRA — países, todas as releases ===')
sample = df_all[
    (df_all['geography'] == 'country') &
    (df_all['facet']     == 'country') &
    (df_all['variable']  == 'usage_per_capita_index') &
    (df_all['geo_id'].isin(COMPARABLES))
][['release_label', 'geo_id', 'country_name', 'variable', 'value']].sort_values(['geo_id', 'release_label'])

display(sample)

In [ ]:
print('=== VARIÁVEIS POR RELEASE ===')
var_avail = (
    df_all.groupby(['release_label', 'variable'])
    .size()
    .reset_index(name='rows')
    .pivot(index='variable', columns='release_label', values='rows')
    .fillna(0)
    .astype(int)
)
display(var_avail)

In [ ]:
print('=== PAÍSES COM DADOS (usage_per_capita_index) por release ===')
country_counts = (
    df_all[
        (df_all['geography'] == 'country') &
        (df_all['variable']  == 'usage_per_capita_index')
    ]
    .groupby('release_label')['geo_id']
    .nunique()
)
print(country_counts.to_string())

---
## 5. Dicionário de Dados

In [ ]:
print('=== COLUNAS DO DATAFRAME ===')
rows = []
for col, meta in BASE_COLUMNS.items():
    rows.append({
        'Coluna':     col,
        'Tipo':       meta.get('type', ''),
        'Descrição':  meta.get('description', ''),
        'Exemplo':    str(meta.get('example', '')),
        'Origem':     meta.get('source', 'Dataset original'),
    })
df_dict_cols = pd.DataFrame(rows)
display(df_dict_cols.style.set_properties(**{'text-align': 'left'}))

In [ ]:
print('=== VARIÁVEIS (coluna variable) ===')
rows = []
for var, meta in VARIABLES.items():
    rows.append({
        'variable':      var,
        'unit':          meta.get('unit', ''),
        'description':   meta.get('description', ''),
        'facets':        ', '.join(meta.get('facets', [])),
        'availability':  ', '.join(meta.get('availability', [])),
        'interpretation': meta.get('interpretation', ''),
    })
df_dict_vars = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 120)
display(df_dict_vars)

In [ ]:
print('=== PADRÕES DE COLABORAÇÃO ===')
for pattern, desc in COLLABORATION_PATTERNS.items():
    print(f'  {pattern:20} {desc}')

---
## 6. Salvar datasets limpos

In [ ]:
# Dataset completo (todas as releases)
out_all = DATA_DIR / 'aei_all_releases.parquet'
df_all.to_parquet(out_all, index=False)
print(f'Salvo: {out_all} ({out_all.stat().st_size / 1e6:.1f} MB)')

# Release enriched (Ago 2025) — única com automation_pct, onet_task_pct_index
df_aug25 = df_all[df_all['release_label'] == 'Ago 2025'].copy()
out_aug = DATA_DIR / 'aei_aug2025.parquet'
df_aug25.to_parquet(out_aug, index=False)
print(f'Salvo: {out_aug} ({out_aug.stat().st_size / 1e6:.1f} MB)')

# Release mais recente (Fev 2026)
df_feb26 = df_all[df_all['release_label'] == 'Fev 2026'].copy()
out_feb = DATA_DIR / 'aei_feb2026.parquet'
df_feb26.to_parquet(out_feb, index=False)
print(f'Salvo: {out_feb} ({out_feb.stat().st_size / 1e6:.1f} MB)')

print('\nPara carregar em outros notebooks:')
print("  df_all   = pd.read_parquet('../data/aei_all_releases.parquet')")
print("  df_aug25 = pd.read_parquet('../data/aei_aug2025.parquet')")
print("  df_feb26 = pd.read_parquet('../data/aei_feb2026.parquet')")

---
## 7. Checklist de qualidade

In [ ]:
checks = [
    ('Todas as 3 releases presentes',
     df_all['release_label'].nunique() == 3),
    ('Brasil (BRA) presente em todas as releases',
     df_all[df_all['geo_id'] == BRAZIL]['release_label'].nunique() == 3),
    ('Coluna country_name preenchida para países reais',
     df_all[df_all['geography'] == 'country']['country_name'].notna().mean() > 0.8),
    ('usage_per_capita_index calculado para todas as releases',
     df_all[df_all['variable'] == 'usage_per_capita_index']['release_label'].nunique() == 3),
    ('automation_pct disponível em Ago 2025',
     len(df_all[(df_all['release_label'] == 'Ago 2025') & (df_all['variable'] == 'automation_pct')]) > 0),
    ('human_only_ability_pct disponível em Fev 2026',
     len(df_all[(df_all['release_label'] == 'Fev 2026') & (df_all['variable'] == 'human_only_ability_pct')]) > 0),
    ('Parquet completo salvo',
     (DATA_DIR / 'aei_all_releases.parquet').exists()),
]

all_pass = True
for label, result in checks:
    icon = 'OK' if result else 'FALHOU'
    print(f'  [{icon}] {label}')
    if not result:
        all_pass = False

print()
print('Setup completo! Pode rodar os demais notebooks.' if all_pass
      else 'Alguns checks falharam — revise antes de continuar.')